# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a [FAIR\^2 Croissant dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect core information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
List available record sets in the dataset, and the fields (columns) for each. All Croissant entities are referenced by their `@id`.

In [ ]:
# List all record set @ids and their fields/columns
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
print("Available Record Sets (@id):\n")
for rs in dataset.metadata.to_json().get('recordSet', []):
    print(f"- RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if fields and isinstance(fields, list):
        print("  Fields/columns (by @id):")
        for field in fields:
            print(f"    - {field['@id']} (name: {field.get('name', 'N/A')})")
    print()
if len(record_set_ids) == 0:
    print("No explicit record sets in top-level metadata, inspecting sources via DataFiles.")
    # Fallback: infer from distributions if recordSet missing (common pattern)
    distros = dataset.metadata.to_json().get('distribution', [])
    for dist in distros:
        print(f"Distribution @id: {dist['@id']}")
        print("You can investigate this distribution as a data source.")

## 3. Data Extraction
To extract records, choose a record set `@id` from above—if none, use the default or only available data file. 

We'll infer the main record set and load the data into a DataFrame.

In [ ]:
# If record sets are defined, use them; else, fallback to the only available table records
if record_set_ids:
    main_record_set = record_set_ids[0]  # You can manually select if >1
    print(f"Using record set: {main_record_set}")
    records = list(dataset.records(record_set=main_record_set))
else:
    # No recordSet entry, use default which loads from primary table
    main_record_set = None
    records = list(dataset.records())

df = pd.DataFrame(records)
print(f"Loaded {len(df)} records. Example columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (such as protein abundance, molecular weight, or coverage percentage) by its column name (which matches the `@id`/name from the schema), filter rows, and normalize it for further analysis.

In [ ]:
# Specify a numeric field available in this dataset. Example: 'MW [kDa]' or 'coverage [%]' or a similar field
for col in df.columns:
    print(col)
# Example common columns may include 'MW [kDa]' (molecular weight), 'coverage [%]', or sample abundance fields
numeric_field = None
for candidate in ['MW [kDa]', 'coverage [%]', 'Peptide count', 'Peptide Count']:
    if candidate in df.columns:
        numeric_field = candidate
        break
if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback

print(f"Selected numeric field for analysis: {numeric_field}")

# Drop rows with missing/NA in this field
filtered_df = df.dropna(subset=[numeric_field])

# Filter by a threshold; use mean if range unknown
threshold = filtered_df[numeric_field].mean()
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize the numeric field to zero mean, unit variance
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a relevant categorical field, e.g., 'Description' or 'Sample' if present
group_candidates = ['Description', 'Protein', 'Sample']
group_field = next((c for c in group_candidates if c in filtered_df.columns), None)
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and optionally, its normalized form.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field before and after normalization
fig, axs = plt.subplots(1, 2, figsize=(14,5))
sns.histplot(filtered_df[numeric_field], kde=True, ax=axs[0], color='skyblue')
axs[0].set_title(f"Distribution of {numeric_field}")
sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, ax=axs[1], color='salmon')
axs[1].set_title(f"Distribution of Normalized {numeric_field}")
plt.show()

# If a group_field exists, show a boxplot
if group_field:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} distribution by {group_field}")
    plt.xticks(rotation=60)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset of human proteins from extracellular vesicles. 

- We loaded the Croissant metadata and identified available record sets and fields by their `@id`.
- We extracted the main table, selected a numeric field for analysis, and demonstrated both normalization and simple filtering.
- We visualized distributions and, if available, grouped the data by categorical fields to examine patterns.

This process showcases a reproducible workflow using `mlcroissant` for FAIR-compliant scientific data packages.

For further work, consider exploring multiple record sets, joining across fields by `@id`, or integrating external protein annotation resources.